# TODO：研究タイトル

**G2：国際紛争の抑止と協調戦略のシミュレーション**（補遺トピック）

チーム名：TODO　／　メンバー：TODO

> **補遺トピック G2・教育用の抽象的な仮想モデル**です。国家（Country A〜E）・利得行列・状態遷移確率・係数はすべて架空で、現実の国際関係・紛争や特定国家の行動を予測するものではありません。扱うのは協調・対話・信頼形成・緊張緩和だけです。

## 使い方（発表前に読む・スライドには出ません）

1. 「準備」セルから順に**すべてのセルを実行**する（グラフのコードは隠してあります）
2. **TODO** をチームの内容に書き換える。**実験していない結果・考察を書かない**
3. HTML スライド：`jupyter nbconvert notebooks/06_research_presentation.ipynb --to slides --no-input --execute`

## 準備：実験環境（最初に必ず実行）

G2 リポジトリの `.py`（計算の本体）を import できるように、作業ディレクトリ（G2 のルート）を自動で探して設定します。

| 実行場所 | 準備 |
| --- | --- |
| ローカルの Jupyter / VSCode | `notebooks/` の中で開けば、そのまま動きます |
| Colab（方法A：GitHub から clone） | 下のセルの `REPO_URL` に**自分たちのリポジトリの URL** を入れる |
| Colab（方法B：ZIP をアップロード） | G2 フォルダを ZIP にして `/content` にアップロードしておく（またはセルの実行中にアップロード） |

Colab の CPU ランタイムで動きます（GPU・TPU は不要）。`textbook/`（教科書のコード）は公開リポジトリに含めない約束です。`textbook/` がないと import が失敗します。

In [ ]:
#@title 準備（G2 の .py を読み込む）
import os, sys, glob, zipfile, subprocess
from pathlib import Path

# ---- 方法A：GitHub から clone する場合だけ設定する（URL は自分たちのものを入れる）----
REPO_URL = ""                                         # 例："https://github.com/<team>/<repo>.git"（空なら clone しない）
REPO_SUBDIR = "g2-conflict-cooperation-simulation"    # リポジトリの中の G2 フォルダ（ルートが G2 なら ""）

IN_COLAB = "google.colab" in sys.modules


def is_g2_root(p):
    return (Path(p) / "game_theory.py").exists() and (Path(p) / "network_model.py").exists()


def find_g2_root():
    for p in [Path.cwd(), Path.cwd().parent]:          # ローカル：notebooks/ の中、または G2 のルート
        if is_g2_root(p):
            return p.resolve()
    if not IN_COLAB:
        return None
    if REPO_URL:                                        # Colab 方法A
        dest = Path("/content/repo")
        if not dest.exists():
            subprocess.run(["git", "clone", "--depth", "1", REPO_URL, str(dest)], check=True)
        cand = dest / REPO_SUBDIR if REPO_SUBDIR else dest
        if is_g2_root(cand):
            return cand
    zips = glob.glob("/content/*.zip")                  # Colab 方法B
    if not zips:
        from google.colab import files
        print("G2 フォルダの ZIP を選んでアップロードしてください")
        zips = ["/content/" + name for name in files.upload()]
    for z in zips:
        with zipfile.ZipFile(z) as zf:
            zf.extractall("/content/g2_upload")
    for hit in glob.glob("/content/g2_upload/**/game_theory.py", recursive=True):
        if is_g2_root(Path(hit).parent):
            return Path(hit).parent
    return None


G2_ROOT = find_g2_root()
if G2_ROOT is None:
    raise RuntimeError("G2 のフォルダが見つかりません。上の説明（方法A／方法B）を確認してください。")
os.chdir(G2_ROOT)
if str(G2_ROOT) not in sys.path:
    sys.path.insert(0, str(G2_ROOT))
if not (G2_ROOT / "textbook" / "fixed_stack.py").exists():
    print("⚠️ textbook/ がありません。network_model.py などの import が失敗します。")

# ---- 既存の .py（計算の本体）と、グラフ ----
import matplotlib.pyplot as plt
import agents as ag
import game_theory as gt
import trust_model as tm
import state_transitions as st
import simulation as sim
import network_model as nm
import cooperation_policies as cp
import evaluation as ev
import conflict_charts as cc

cc.setup_style()                        # 文字を大きくするなら cc.setup_style(1.3)
COUNTRIES = ag.load_countries()         # 架空の国家 A〜E
PARAMS = sim.load_parameters()          # シミュレーションのパラメータ（すべて架空）
M = gt.load_payoff_matrix()             # 利得行列（架空）
IDS = [c["id"] for c in COUNTRIES]
RUNS = PARAMS["monte_carlo_runs"]       # モンテカルロの反復回数
YEARS = PARAMS["simulation_years"]
LABELS = {s: gt.label(s) for s in ag.STRATEGIES}

SAVE_FIGURES = False                    # True にすると、グラフを notebooks/figures/ に PNG で保存する
FIG_DIR = "notebooks/figures"


def show(fig, name):
    """グラフを表示し、SAVE_FIGURES = True なら PNG も保存する"""
    if SAVE_FIGURES:
        print("saved:", cc.save(fig, f"{FIG_DIR}/{name}.png"))
    plt.show()


def table(rows, columns):
    """辞書のリストを表で表示する。columns = [(キー, 見出し, 書式)]。pandas があれば DataFrame で表示"""
    try:
        import pandas as pd
        from IPython.display import display
        display(pd.DataFrame([{h: ("-" if r.get(k) is None else f.format(r[k])) for k, h, f in columns} for r in rows]))
    except ImportError:
        print(ev.format_table(rows, columns))


SUMMARY_COLUMNS = [("name", "condition", "{}"), ("cooperation_rate", "cooperation", "{:.3f}"),
                   ("cooperation_rate_sd", "sd", "{:.3f}"), ("mutual_cooperation_rate", "mutual coop.", "{:.3f}"),
                   ("cooperation_state_rate", "Cooperation state", "{:.3f}"), ("tension_rate", "Tension", "{:.3f}"),
                   ("conflict_rate", "Conflict", "{:.3f}"), ("conflict_rate_sd", "sd", "{:.3f}")]

print("G2_ROOT:", G2_ROOT)
print("架空の国家:", ", ".join(f"{c['id']}({LABELS[c['strategy']]})" for c in COUNTRIES))
print(f"モンテカルロ {RUNS} 回 × {YEARS} 年、乱数シード {PARAMS['random_seed']}")

## 研究背景

- TODO：なぜ協調・信頼形成・緊張緩和をアルゴリズムで考えるのか

## Research Question

- TODO：チームが選んだ RQ（RQ1〜RQ6 から）

## 仮説

- TODO：**実験の前に**立てた仮説と、その理由

## ゲーム理論モデル

- 2 国の協調（C）・非協調（N）を 2×2 の利得行列で表す（囚人のジレンマ型、架空の値）
- 一回限りのゲームの Nash 均衡と、繰り返しゲームでの協調は別の話

In [ ]:
#@title 利得行列
show(cc.plot_payoff_matrix(M), "06_payoff")

## 状態遷移モデル

- Cooperation・Neutral・Tension・Conflict の 4 状態（抽象的なラベル）
- 次の状態は行動・信頼度・対話・協定・隣からの波及・外部ショックで変わる確率で決まる（架空の値）

In [ ]:
#@title 状態遷移図
show(cc.plot_transition_diagram(st.transition_matrix("C", "N", 0.5, False, False, PARAMS["transition"]), st.STATES,
                                "State transitions: one-sided cooperation (model)"), "06_diagram")

## 信頼度モデル

- 信頼度（0〜1）は、相互協調で上がり、非協調で下がる。対話・協定で少し上がる（係数は架空）
- 信頼度が高くても、必ず協調するとは限らない

## ネットワークモデル

- 国家 = ノード、関係 = エッジ（無向グラフ）。Line・Ring・Star・Random
- DFS（スタック）と BFS（キュー）で到達可能性を調べる（どちらも O(V + E)）

In [ ]:
#@title ネットワーク
NETS = {k: nm.make_network(k, IDS, PARAMS["network"]["random_edge_probability"], PARAMS["network"]["random_seed"])
        for k in nm.NETWORK_TYPES}
show(cc.plot_networks([(k.capitalize(), IDS, e) for k, e in NETS.items()], "Network structures"), "06_networks")

## 比較する戦略

| 戦略 | 行動の決め方 |
| --- | --- |
| Always Cooperate | 常に協調 |
| Always Non-cooperate | 常に非協調 |
| Tit-for-Tat | 初回は協調、以後は相手の前回の行動 |
| Conditional | 信頼度・相手の最近の協調率などで決まる確率で協調 |

## 実験条件

- TODO：ネットワーク・国家・試行回数・年数・変えたパラメータをチームの実験に合わせて書く

In [ ]:
#@title 実験条件
EDGES = NETS["ring"]                 # TODO：チームの実験条件にする
print(f"5 国・リング、モンテカルロ {RUNS} 回 × {YEARS} 年、乱数シード {PARAMS['random_seed']}")
strat = {s: sim.monte_carlo(ag.with_strategy(COUNTRIES, s), EDGES, PARAMS, M, RUNS, years=YEARS)["summary"]
         for s in ag.STRATEGIES}

## 結果：協調率

In [ ]:
#@title 協調率
names = [LABELS[s] for s in ag.STRATEGIES]
show(cc.plot_bar(names, [strat[s]["cooperation_rate"] for s in ag.STRATEGIES],
                 "Cooperation rate when all countries use the same strategy (model)", "cooperation rate",
                 [cc.STRATEGY_COLORS[s] for s in ag.STRATEGIES], "{:.3f}",
                 [strat[s]["cooperation_rate_sd"] for s in ag.STRATEGIES], (0, 1.15)), "06_coop")

## 結果：紛争状態率

In [ ]:
#@title 紛争状態率
show(cc.plot_bar(names, [strat[s]["conflict_rate"] for s in ag.STRATEGIES],
                 "Conflict-state rate when all countries use the same strategy (model)", "conflict-state rate",
                 [cc.STRATEGY_COLORS[s] for s in ag.STRATEGIES], "{:.3f}",
                 [strat[s]["conflict_rate_sd"] for s in ag.STRATEGIES], (0, 1.0)), "06_conflict")

## 結果：最適化アルゴリズムの比較

In [ ]:
#@title 全探索と Greedy 法
res = cp.compare(COUNTRIES, EDGES, PARAMS, M)
table([{"name": "No support", "objective": res["baseline"]}] +
      [{"name": x["algorithm"], "objective": x["objective"], "gap": x["gap"], "evals": x["evaluations"],
        "runtime": x["runtime_s"], "selected": " ".join(f"{u}-{v}" for u, v in x["selected"])}
       for x in (res["exhaustive"], res["greedy"])],
      [("name", "algorithm", "{}"), ("objective", "objective", "{:.4f}"), ("gap", "gap", "{:.4f}"),
       ("evals", "evaluations", "{}"), ("runtime", "runtime (s)", "{:.2f}"), ("selected", "supported pairs", "{}")])
show(cc.plot_networks([("Exhaustive", IDS, EDGES), ("Greedy", IDS, EDGES)], "Supported pairs",
                      highlight=[res["exhaustive"]["selected"], res["greedy"]["selected"]]), "06_supported")

## 感度分析

- TODO：チームの感度分析のグラフに置き換える

In [ ]:
#@title 感度分析
# TODO：チームの感度分析に置き換える（例：対話の有無と対照条件）
control = sim.with_changes(PARAMS, **{"transition.dialogue_affects_transitions": False})
conds = {"no dialogue": (PARAMS, False), "dialogue": (PARAMS, True), "dialogue (control)": (control, True)}
summ = {n: sim.monte_carlo(COUNTRIES, EDGES, p, M, RUNS, years=YEARS, dialogue=d)["summary"] for n, (p, d) in conds.items()}
keys = ["cooperation_state_rate", "tension_rate", "conflict_rate"]
show(cc.plot_grouped(list(conds), keys, {k: [summ[n][k] for n in conds] for k in keys},
                     "Dialogue and the control model (model)", "share",
                     colors={"cooperation_state_rate": cc.STATE_COLORS["Cooperation"], "tension_rate": cc.STATE_COLORS["Tension"],
                             "conflict_rate": cc.STATE_COLORS["Conflict"]}, ylim=(0, 1.0)), "06_sensitivity")

## 考察

- TODO：仮説は支持されたか。**どのモデルの仮定の下で**そう言えるか
- TODO：アルゴリズムの比較から分かること（最適性・計算量・ネットワークの大きさ）

## モデルの限界

- 教育用の**抽象的な仮想モデル**。実際の国家の行動や国際紛争の発生確率を予測・推定しない
- 利得行列・状態遷移確率・係数は架空。対話や協定の効果はモデルの**仮定**に依存する
- ゲーム理論上の均衡と現実の外交的安定性を混同しない。ネットワークの接続性と政治的な協調関係を同一視しない
- 国家間の関係を過度に単純化している。モデル上の最適戦略を現実の政策判断に直接使わない
- TODO：チームが気づいた限界を追加する

## 結論

- TODO：RQ への答えを、**仮定とともに**書く

## 今後の課題

- TODO：Advanced A〜G のどれに取り組むか（繰り返しゲームの発展、適応的戦略、多国間協定、ネットワークの変化、確率的ショック、多目的最適化、強化学習）

## 参考文献

- 柴田望洋『新・明解 Pythonで学ぶアルゴリズムとデータ構造（第2版）』SBクリエイティブ（第2章 最大値、第3章 線形探索、第4章 スタック・キュー、第5章 再帰、第6章 単純挿入ソート）
- R. Axelrod, *The Evolution of Cooperation*, Basic Books, 1984（繰り返し囚人のジレンマと Tit-for-Tat）
- G2 リポジトリの `README.md`（モデルの仮定・計算量・モデルの限界と倫理的配慮）
- データ：`data/sample_countries.json`、`data/payoff_matrix.json`、`data/simulation_parameters.json`（この教材のための**架空のデータ**）